# Fine-tune Whisper on CDLI Non-Standard Kenyan Speech (Noise-Robust)

This notebook fine-tunes Whisper on CDLI non-standard Kenyan speech datasets with **waveform-level audio augmentation** to improve robustness in real-world noisy environments (ambient crowd noise, GSM codec compression, room reverb).

**Changes from baseline notebook:**
- `prepare_features_augmented()` applies waveform augmentation on the training split only
- `augment_audio()` simulates: volume perturbation, Gaussian noise, GSM codec downsampling, and room reverb
- `AUGMENT_PROB` controls augmentation probability per example (default 0.5)
- Evaluation uses clean audio throughout for comparable WER/CER
- `processing_class` replaces deprecated `tokenizer` in Trainer
- Early stopping and weight decay consolidated into `Seq2SeqTrainingArguments`

## Authentication

In [ ]:
from huggingface_hub import login
HF_TOKEN = input("Enter HF token: ")
login(token=HF_TOKEN)

## Settings

**Adapt these for your run. Everything else should be left as-is.**

### Directories

In [ ]:
import os

LOCAL_STORAGE_DIR = '/jupyter_kernel'
BASE_DIR = os.path.join(LOCAL_STORAGE_DIR, 'trained_models')
os.makedirs(BASE_DIR, exist_ok=True)

# Set run name -- increment for each new run
OUTPUT_DIR = os.path.join(BASE_DIR, 'whisper-small-kenyan-english-nonstandard-robust_v1_run6')
# OUTPUT_DIR = os.path.join(BASE_DIR, 'whisper-small-kenyan-swahili-nonstandard-robust_v1_run1')

print(f"Will write model to: {OUTPUT_DIR}")
if os.path.exists(OUTPUT_DIR):
    raise ValueError("Output directory already exists. Increment run number or delete existing directory.")


### Model and Dataset Settings

In [ ]:
WHISPER_MODEL_TYPE = "openai/whisper-small"
# WHISPER_MODEL_TYPE = "openai/whisper-medium"

# English non-standard
LANGUAGE = 'en'
DATASET_NAME = "cdli/kenyan_english_nonstandard_speech_v1.0"


### Augmentation Settings

Controls waveform-level noise augmentation applied during training only.
Start with `AUGMENT_PROB = 0.5`. Increase to 0.7 in later runs if WER improves.

In [ ]:
# Master switch: set to False to disable all waveform augmentation (reproduces baseline)
USE_WAVEFORM_AUGMENTATION = True

# Probability that any single augmentation is applied to a given example
# Applies only to the augmented half of the train set when USE_50_50_BATCH_MIX=True
AUGMENT_PROB = 0.4

# Run 5: structured 50/50 clean/noisy training mix.
# Instead of a per-example random augmentation coin flip across the whole train set
# (Runs 1-4), the train set is split into a clean half and an augmented half and
# concatenated. This guarantees a stable, consistent noisy exposure every epoch.
USE_50_50_BATCH_MIX = False

# Individual augmentation enable flags
AUG_VOLUME_PERTURB = True   # always mild, safe to keep on
AUG_GAUSSIAN_NOISE = True   # simulates crowd/ambient noise
AUG_GSM_CODEC      = True   # simulates mobile network compression
AUG_REVERB         = True # simulates small room/kiosk acoustics

# Noise intensity bounds (increase cautiously for larger datasets)
NOISE_LEVEL_MIN = 0.002
NOISE_LEVEL_MAX = 0.01


### Model Architecture Settings

In [ ]:
# Which parts of the model to update
UPDATE_ENCODER = True
UPDATE_PROJ    = True
UPDATE_DECODER = True  # set False to use partial decoder unfreezing below

# Partial decoder unfreezing (only applies when UPDATE_DECODER = False)
# whisper-small has 12 decoder layers (0-11). Unfreeze the last N.
NUM_DECODER_LAYERS_TO_UNFREEZE = 2

# SpecAugment (operates on log-mel spectrograms, complementary to waveform augmentation)
USE_SPECAUGMENT = True

### Trainer Settings

In [ ]:
LOGGING_STEPS = 5
SAVE_STEPS    = 50   # set to 0 to only save last and best

MAX_EPOCHS = 10
MAX_STEPS  = 2000    # revert from 2500 — back to Run 3's budget
                      # train set size under the 50/50 mix

LEARNING_RATE     = 3e-6
LR_SCHEDULER_TYPE = 'polynomial'   # 'constant_with_warmup' or 'polynomial'
LR_WARMUP_STEPS   = 100
LR_END            = 1e-8
LR_DECAY_POWER    = 1

WEIGHT_DECAY            = 0.01
EARLY_STOPPING_PATIENCE = 7

BATCH_SIZE      = 32
EVAL_BATCH_SIZE = 16

MAX_GEN_LEN   = 128
EVAL_ON_START = True
EVAL_STEPS    = 50

USE_FP16 = True
USE_BF16 = False   # enable for A100/A40

NUM_CHECKPOINTS_TO_STORE = 2

# Run 4 finding: switching the early stopping signal to noisy dev WER alone
# (with no change to training data) produced no measurable robustness gain.
# Reverted to clean dev WER for Run 5 so the only variable under test is the
# 50/50 training data mix.
USE_NOISY_DEV_FOR_EARLY_STOPPING = False

# Don't change these
TASK            = "transcribe"
BASE_MODEL_NAME = WHISPER_MODEL_TYPE
print(f"Base model: {BASE_MODEL_NAME} | Language: {LANGUAGE} | Dataset: {DATASET_NAME}")
print(f"Waveform augmentation: {USE_WAVEFORM_AUGMENTATION} | Augment prob: {AUGMENT_PROB}")
print(f"50/50 batch mix: {USE_50_50_BATCH_MIX}")
print(f"Noisy dev early stopping: {USE_NOISY_DEV_FOR_EARLY_STOPPING}")


## Imports and Environment Setup

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torchaudio
import torchaudio.transforms as T
import librosa
import datasets
import evaluate
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Any, Dict, List, Union

from huggingface_hub import hf_hub_download
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# Disable dataset caching (saves disk on Modal volumes)
datasets.disable_caching()
print('Dataset caching:', datasets.is_caching_enabled())

# Avoid thread contention with multiprocessing map
torch.set_num_threads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

num_proc = min(32, os.cpu_count())
print(f"CPU workers for dataset mapping: {num_proc}")

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")
transcript_normalizer = BasicTextNormalizer()

## Waveform Augmentation

Applied to training audio only, before log-mel feature extraction.
Simulates four real-world Kenyan deployment conditions:

1. **Volume perturbation** — microphone gain variation across devices
2. **Gaussian noise** — ambient crowd noise, wind, market environments
3. **GSM codec simulation** — 8kHz downsample/resample to mimic mobile network compression
4. **Room reverb** — small kiosk/office acoustic echo

Evaluation splits always use clean audio for reproducible WER/CER comparisons.

In [ ]:
def augment_audio(audio_array: np.ndarray, sample_rate: int, apply_prob: float = 0.5) -> np.ndarray:
    """
    Apply waveform-level augmentations to simulate Kenyan deployment conditions.

    Args:
        audio_array: Raw audio waveform as numpy array.
        sample_rate: Audio sample rate (typically 16000 Hz).
        apply_prob: Probability that each stochastic augmentation is applied.

    Returns:
        Augmented waveform as numpy array, same shape as input.
    """
    waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0)  # (1, T)

    # 1. Volume perturbation (always applied, mild range)
    if AUG_VOLUME_PERTURB:
        gain = random.uniform(0.7, 1.3)
        waveform = waveform * gain

    # 2. Gaussian noise (crowd/ambient)
    if AUG_GAUSSIAN_NOISE and random.random() < apply_prob:
        noise_level = random.uniform(NOISE_LEVEL_MIN, NOISE_LEVEL_MAX)
        noise = torch.randn_like(waveform) * noise_level
        waveform = waveform + noise

    # 3. GSM codec simulation (mobile network compression artifact)
    # Downsample to 8kHz then resample back to original rate
    if AUG_GSM_CODEC and random.random() < apply_prob:
        resample_down = T.Resample(orig_freq=sample_rate, new_freq=8000)
        resample_up   = T.Resample(orig_freq=8000, new_freq=sample_rate)
        waveform = resample_up(resample_down(waveform))

    # 4. Room reverb (small kiosk/office acoustic echo)
    # Implemented as a short delay blend; lower prob since it's more disruptive
    if AUG_REVERB and random.random() < apply_prob * 0.5:
        reverb_gain   = random.uniform(0.1, 0.25)
        delay_samples = random.randint(int(0.01 * sample_rate), int(0.05 * sample_rate))
        delayed = torch.zeros_like(waveform)
        delayed[:, delay_samples:] = waveform[:, :-delay_samples]
        waveform = waveform + reverb_gain * delayed

    # Clip to prevent saturation from compounded augmentations
    waveform = torch.clamp(waveform, -1.0, 1.0)
    return waveform.squeeze(0).numpy()

## Helper Functions

In [ ]:
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_dataset_split(dataset_name: str, split: str, limit_to_30_seconds: bool = True):
    """Load a HF dataset split and optionally filter to Whisper's 30s context window."""
    if split not in ['train', 'test', 'validation']:
        raise ValueError("split must be one of 'train', 'test', or 'validation'")
    ds = datasets.load_dataset(dataset_name, split=split, streaming=False)
    orig_len = len(ds)
    if limit_to_30_seconds:
        ds = ds.filter(lambda ex: ex['audio_length'] <= 30)
        print(f"[{split}] Filtered {orig_len} -> {len(ds)} examples (<= 30s)")
    return ds


def is_valid_label_length(example):
    """Exclude examples with token sequences exceeding Whisper's decoder limit."""
    if 'labels' not in example:
        return False
    valid_labels = [l for l in example['labels'] if l != -100]
    return len(valid_labels) <= 448


def get_wer(references, predictions, normalize=True, verbose=True):
    rs, ps = references, predictions
    if normalize:
        ps = [transcript_normalizer(x) for x in predictions]
        rs = [transcript_normalizer(x) for x in references]
    if verbose:
        for r, p in zip(rs, ps):
            print(f"REF: {r}")
            print(f"HYP: {p}")
            print()
    return wer_metric.compute(references=rs, predictions=ps)


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_strs  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_strs = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wers, cers = [], []
    for pred_str, label_str in zip(pred_strs, label_strs):
        p = transcript_normalizer(pred_str)
        l = transcript_normalizer(label_str)
        wers.append(wer_metric.compute(predictions=[p], references=[l]))
        cers.append(cer_metric.compute(predictions=[p], references=[l]))

    wer = np.mean([min(1.0, x) for x in wers])
    cer = np.mean([min(1.0, x) for x in cers])
    print(f'WER (adjusted): {wer:.4f} | CER (adjusted): {cer:.4f}')
    return {"wer": wer, "cer": cer}

## Feature Extraction Functions

Two variants:
- `prepare_features` — clean extraction for eval/test splits
- `prepare_features_augmented` — waveform augmentation then extraction for training split

In [ ]:
print(f"Loading processor for: {WHISPER_MODEL_TYPE} | language: {LANGUAGE}")
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_TYPE, language=LANGUAGE, task=TASK)


def prepare_features(example):
    """Clean feature extraction. Use for dev and test splits."""
    example["input_features"] = processor.feature_extractor(
        example["audio"]["array"],
        sampling_rate=example["audio"]["sampling_rate"]
    ).input_features[0]
    example["labels"]       = processor.tokenizer(example["transcription"]).input_ids
    example["token_length"] = len(example["labels"])
    return example


def prepare_features_augmented(example):
    """
    Waveform augmentation then feature extraction. Use for training split only.
    Falls back to clean extraction if USE_WAVEFORM_AUGMENTATION is False.
    """
    audio_array = example["audio"]["array"]
    sample_rate = example["audio"]["sampling_rate"]

    if USE_WAVEFORM_AUGMENTATION:
        audio_array = augment_audio(audio_array, sample_rate, apply_prob=AUGMENT_PROB)

    example["input_features"] = processor.feature_extractor(
        audio_array, sampling_rate=sample_rate
    ).input_features[0]
    example["labels"]       = processor.tokenizer(example["transcription"]).input_ids
    example["token_length"] = len(example["labels"])
    return example

## Data Collator

In [ ]:
# Note: "The attention mask is not set..." warning on this collator can be safely ignored.
# See: https://discuss.huggingface.co/t/finetuning-whisper-attention-mask-not-set-and-canot-be-inferred/97456

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    # Alias so newer Trainer versions don't fall back to the deprecated path
    @property
    def tokenizer(self):
        return self.processor.tokenizer

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels         = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

## Load and Prepare Datasets

Dev and test splits use `prepare_features` (clean audio) for reporting.

Run 5 change: the training split is no longer built with a single call to
`prepare_features_augmented` (which applies augmentation via a random per-example
coin flip across the whole set). Instead, when `USE_50_50_BATCH_MIX=True`, the train
split is loaded twice — once through `prepare_features` (clean) and once through
`prepare_features_augmented` (augmented) — and the two halves are concatenated and
shuffled. This guarantees a stable 50% clean / 50% noisy exposure every epoch, rather
than the variable ~60/40 split that resulted from the random coin flip at
`AUGMENT_PROB=0.4` in Runs 1-4.

Early stopping reverted to clean dev WER (Run 3 behaviour) per the Run 4 findings —
see Trainer Settings above.


In [ ]:
if USE_50_50_BATCH_MIX:
    # Clean half
    train_clean = load_dataset_split(DATASET_NAME, split='train', limit_to_30_seconds=True)
    train_clean = train_clean.map(
        prepare_features,
        remove_columns=['audio'],
        writer_batch_size=1,
        num_proc=num_proc
    )
    print(f"Clean train half: {len(train_clean)}")

    # Augmented half
    train_noisy = load_dataset_split(DATASET_NAME, split='train', limit_to_30_seconds=True)
    train_noisy = train_noisy.map(
        prepare_features_augmented,
        remove_columns=['audio'],
        writer_batch_size=1,
        num_proc=num_proc
    )
    print(f"Augmented train half: {len(train_noisy)}")

    import datasets as hf_datasets
    train_dataset = hf_datasets.concatenate_datasets([train_clean, train_noisy]).shuffle(seed=42)
    print(f"50/50 mix train size: {len(train_dataset)}")
else:
    train_dataset = load_dataset_split(DATASET_NAME, split='train', limit_to_30_seconds=True)
    train_dataset = train_dataset.map(
        prepare_features_augmented,
        remove_columns=['audio'],
        writer_batch_size=1,
        num_proc=num_proc
    )
    print(f"Train examples after processing: {len(train_dataset)}")


In [ ]:
# Clean dev set — used both for early stopping (Run 5 reverts to clean dev) and reporting
dev_dataset = load_dataset_split(DATASET_NAME, split='validation', limit_to_30_seconds=True)
dev_dataset = dev_dataset.map(
    prepare_features,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc
)
print(f"Dev examples after processing: {len(dev_dataset)}")

# Optional noisy dev mirror, retained for parity with Run 4 in case it's needed
# for diagnostics. Not used by the Trainer in Run 5 (USE_NOISY_DEV_FOR_EARLY_STOPPING=False).
if USE_NOISY_DEV_FOR_EARLY_STOPPING:
    noisy_dev_dataset = load_dataset_split(DATASET_NAME, split='validation', limit_to_30_seconds=True)
    noisy_dev_dataset = noisy_dev_dataset.map(
        prepare_features_augmented,
        remove_columns=['audio'],
        writer_batch_size=1,
        num_proc=num_proc
    )
    print(f"Noisy dev examples after processing: {len(noisy_dev_dataset)}")
    eval_dataset_for_trainer = noisy_dev_dataset
else:
    eval_dataset_for_trainer = dev_dataset

print(f"Early stopping signal: {'noisy dev' if USE_NOISY_DEV_FOR_EARLY_STOPPING else 'clean dev'}")


In [ ]:
test_dataset = load_dataset_split(DATASET_NAME, split='test', limit_to_30_seconds=True)
test_dataset = test_dataset.map(
    prepare_features,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc
)
print(f"Test examples after processing: {len(test_dataset)}")


## Load and Configure Model

In [ ]:
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_NAME)
base_model = base_model.to(device)

# Task and language
base_model.generation_config.language = LANGUAGE
base_model.generation_config.task     = TASK
base_model.generation_config.forced_decoder_ids = None
base_model.config.forced_decoder_ids  = None
base_model.config.use_cache           = False  # required for gradient checkpointing

print(f"Model loaded: {WHISPER_MODEL_TYPE}")
print(f"Language: {base_model.generation_config.language}")

In [ ]:
if USE_SPECAUGMENT:
    base_model.config.apply_spec_augment     = True
    base_model.config.mask_time_prob         = 0.05
    base_model.config.mask_time_length       = 10
    base_model.config.mask_time_min_masks    = 2
    base_model.config.mask_feature_prob      = 0.05
    base_model.config.mask_feature_length    = 10
    base_model.config.mask_feature_min_masks = 2

print(f"SpecAugment: {base_model.config.apply_spec_augment}")

### Layer Freezing

Run **one** of the two cells below depending on your `UPDATE_DECODER` setting.

In [ ]:
# Full unfreeze (run this when UPDATE_DECODER = True)
if UPDATE_DECODER:
    base_model.model.encoder.requires_grad_(UPDATE_ENCODER)
    base_model.model.decoder.requires_grad_(True)
    base_model.proj_out.requires_grad_(UPDATE_PROJ)

    print(f"Encoder params: {count_trainable_parameters(base_model.model.encoder):,} / {base_model.model.encoder.num_parameters():,}")
    print(f"Decoder params: {count_trainable_parameters(base_model.model.decoder):,} / {base_model.model.decoder.num_parameters():,}")
    print(f"Total trainable: {count_trainable_parameters(base_model):,} / {base_model.model.num_parameters():,}")

In [ ]:
# Partial decoder unfreeze (run this when UPDATE_DECODER = False)
if not UPDATE_DECODER:
    base_model.model.encoder.requires_grad_(UPDATE_ENCODER)
    base_model.proj_out.requires_grad_(UPDATE_PROJ)

    # Freeze full decoder, then selectively unfreeze last N layers
    base_model.model.decoder.requires_grad_(False)
    for layer in base_model.model.decoder.layers[-NUM_DECODER_LAYERS_TO_UNFREEZE:]:
        layer.requires_grad_(True)

    print(f"Partial unfreeze: last {NUM_DECODER_LAYERS_TO_UNFREEZE} decoder layers")
    print(f"Encoder params: {count_trainable_parameters(base_model.model.encoder):,} / {base_model.model.encoder.num_parameters():,}")
    print(f"Decoder params: {count_trainable_parameters(base_model.model.decoder):,} / {base_model.model.decoder.num_parameters():,}")
    print(f"Total trainable: {count_trainable_parameters(base_model):,} / {base_model.model.num_parameters():,}")

## Configure Trainer

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
    logging_steps=LOGGING_STEPS,
    report_to=["tensorboard"],
    include_num_input_tokens_seen=True,
    fp16=USE_FP16,
    bf16=USE_BF16,
    push_to_hub=False,
    remove_unused_columns=False,
    num_train_epochs=MAX_EPOCHS,
    max_steps=MAX_STEPS,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    per_device_train_batch_size=BATCH_SIZE,
    eval_on_start=EVAL_ON_START,
    predict_with_generate=True,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    generation_max_length=MAX_GEN_LEN,
    metric_for_best_model="wer",
    greater_is_better=False,
    load_best_model_at_end=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    lr_scheduler_kwargs={
        "lr_end": LR_END,
        "power": LR_DECAY_POWER,
    },
    learning_rate=LEARNING_RATE,
    warmup_steps=LR_WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=NUM_CHECKPOINTS_TO_STORE,
)

print(f"Trainer configured. Output: {OUTPUT_DIR}")
print(f"Max steps: {MAX_STEPS} | Weight decay: {training_args.weight_decay} | Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Eval dataset: {'noisy dev' if USE_NOISY_DEV_FOR_EARLY_STOPPING else 'clean dev'}")


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=base_model.config.decoder_start_token_id,
)

# eval_dataset_for_trainer: clean dev in Run 5 (USE_NOISY_DEV_FOR_EARLY_STOPPING=False).
trainer = Seq2SeqTrainer(
    args=training_args,
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset_for_trainer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)


## Run Training

TensorBoard logs write to `OUTPUT_DIR/logs`. Use `tensorboard_server.py` on Modal to view.

Run the model dir print below first to copy the path.

In [ ]:
print('Training dir (for TensorBoard):', OUTPUT_DIR)

In [ ]:
# Fresh training run
#trainer.train()

# Resume from checkpoint if interrupted:
trainer.train(resume_from_checkpoint=True)

## Evaluate

Both splits use clean (unaugmented) audio. WER/CER here reflects real-world clean speech performance.

Run 5 uses clean dev WER as the Trainer's eval signal (reverted from Run 4's noisy dev
experiment), so `eval_dataset_for_trainer` and `dev_dataset` are the same data here.
The cells below report clean dev and clean test WER/CER for the best checkpoint.


In [ ]:
print("--- Dev set evaluation (clean) ---")
trainer.evaluate(dev_dataset, language=LANGUAGE)


In [ ]:
print("--- Test set evaluation (clean) ---")
trainer.evaluate(test_dataset, language=LANGUAGE)


## Evaluate on Noise-Augmented Test Set

Runs the same test split through the augmentation pipeline before evaluation.
Comparing these results against the clean test WER quantifies noise robustness.

In Run 4, the best checkpoint was selected based on noisy dev WER, so this evaluation
reflects a checkpoint that was explicitly optimised for noisy conditions — unlike Runs 1–3
where the checkpoint was selected on clean dev WER.

Publish clean test WER, noisy test WER, and the gap (robustness delta) in the model card.


In [ ]:
# Reload raw test split and apply augmented feature extraction
noisy_test_dataset = load_dataset_split(DATASET_NAME, split='test', limit_to_30_seconds=True)
noisy_test_dataset = noisy_test_dataset.map(
    prepare_features_augmented,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc
)
print(f"Noisy test examples: {len(noisy_test_dataset)}")

print("--- Noise-augmented test set evaluation ---")
trainer.evaluate(noisy_test_dataset, language=LANGUAGE)


## Save Model

In [ ]:
# load_best_model_at_end=True means the best checkpoint is already loaded after training.
best_model_dir = os.path.join(OUTPUT_DIR, 'best_model')
print(f"Saving best model to: {best_model_dir}")
trainer.model.save_pretrained(best_model_dir, safe_serialization=True)
processor.save_pretrained(best_model_dir)
print("Done.")